# 13 — Register a future model comparison (no price data loaded)

July–September 2026 exposure is uncertain, so it must not be described as an independent test. This notebook freezes the original 2025 model and the longer-history candidate **before a future outcome period**, without opening later prices or labels.

**Proposed preliminary period:** four complete outcome weeks, **2–29 November 2026**. Forecast-origin weeks are **26 October–22 November**; the first forecast would be issued after **1 November** closes. Registration must occur before 1 November. Evaluate no earlier than **30 November**, once the full outcome data is available. Four weeks provide limited evidence, particularly for rare food groups; this is not a substitute for broader seasonal validation.

Run All using the current completed run after 12. There are no path/mode edits or new dependencies. It freezes the most recently completed 12 candidate compatible with the original model, based on file completion time, never metric ranking. Both upstream identities and the exact copied model hashes are printed/recorded. If this is not the intended candidate, do not evaluate it; clarify the checkpoint while the future period remains untouched.

Only three generated files are needed: two frozen model copies and `registration.json`. No data downloads, predictions, fitting, cutoff selection or evaluation occurs. Existing experiments, dashboard inputs and active-run selection stay unchanged. A repeated Run All verifies the same registration without replacing it. This future plan need not delay dashboard development or an earlier FYP deadline.

In [1]:
# Open this notebook from the FYP folder or its research subfolder.
import sys
from pathlib import Path
REPOSITORY_ROOT = Path.cwd()
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open the FYP project folder before running this research notebook.")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import hashlib
import json
import shutil
from pathlib import Path
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
from importlib import reload, metadata
import pricecatcher_config as cfg
cfg = reload(cfg)

OUTCOME_START = date(2026, 11, 2)
OUTCOME_END = date(2026, 11, 29)
FIRST_FORECAST_CLOSE = OUTCOME_START - timedelta(days=1)
EARLIEST_EVALUATION = OUTCOME_END + timedelta(days=1)
registration_dir = cfg.PROJECT_ROOT / "13_registered_future_evaluation" / f"{OUTCOME_START}_{OUTCOME_END}"
registration_path = registration_dir / "registration.json"

def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

design = {
    "schema_version": 1,
    "purpose": "reserved future comparison of two frozen models; not yet evaluated",
    "outcome_start": str(OUTCOME_START), "outcome_end": str(OUTCOME_END),
    "first_forecast_close": str(FIRST_FORECAST_CLOSE), "earliest_evaluation": str(EARLIEST_EVALUATION),
    "july_september_2026_exposure": "uncertain; do not claim independent validation",
    "primary_metric": "paired difference in average precision on identical eligible rows",
    "secondary_metrics": ["precision", "recall", "F1", "confusion counts", "food-group and weekly breakdowns"],
    "decision_rule": "report both models and all groups; a positive AP difference alone does not establish practical or universal superiority",
    "constraints": ["no refitting, new features, cleaning thresholds, cutoff search or model changes",
        "same evaluation rows and observed 10% next-week target for both models",
        "training-fitted preprocessing only; exact-week past features and frozen item/retail eligibility",
        "missing outcomes stay unlabelled; missing required geography/units are audited, never guessed",
        "do not inspect November outcome prices/labels before the planned evaluation",
        "one reserved-period evaluation; repeated execution is reproduction, not new independent evidence"],
    "limitations": ["four weeks only", "rare positive groups may be unevaluable", "uncalibrated scores",
        "historical lookup assumptions and source revisions remain",
        "registration does not establish real-time source availability or archive live predictions"],
}
if registration_path.exists():
    registration = json.loads(registration_path.read_text())
    content = {k: v for k, v in registration.items() if k != "registration_sha256"}
    if registration["registration_sha256"] != canonical_hash(content) or registration["design"] != design:
        raise ValueError("Registered protocol changed. Restore its original file; do not rewrite the reserved comparison.")
    for name, record in registration["models"].items():
        if not (registration_dir / record["frozen_file"]).is_file() or file_hash(registration_dir / record["frozen_file"]) != record["sha256"]:
            raise ValueError(f"Frozen {name} copy changed. Restore its matching model.")
    print("Verified existing registration; no new candidate selection or replacement.")
else:
    now = datetime.now(ZoneInfo("Asia/Kuala_Lumpur"))
    if now.date() >= FIRST_FORECAST_CLOSE:
        raise ValueError("It is too late to register this proposed period prospectively. Choose an uninspected later period before accessing any outcomes.")
    model_dir = next((p for p in cfg.MODEL_DIRS if (p / "best_model.joblib").is_file()), None)
    if model_dir is None:
        raise FileNotFoundError("Complete 06 and 12 first; their model files must be available locally.")
    manifest = json.loads((model_dir / "model_manifest.json").read_text())
    selection = json.loads((model_dir / "selection.json").read_text())
    original_path = model_dir / "best_model.joblib"
    if (manifest.get("run_status") != "completed" or selection != manifest["selection"]
            or file_hash(original_path) != manifest["model_sha256"]):
        raise ValueError("A verified completed original model and selection are required.")
    settings_path = cfg.PROJECT_ROOT / "experiments" / "holdout" / "05_modelling" / "experiment_settings.json"
    if file_hash(settings_path) != manifest["input_sha256"]["holdout/experiment_settings.json"]:
        raise ValueError("Original modelling settings changed. Restore the frozen version.")
    settings = json.loads(settings_path.read_text())
    cleaning_path = settings_path.parent.parent / "03_cleaned" / "cleaning_summary.json"
    cleaning = json.loads(cleaning_path.read_text())
    if cleaning["preparation"] != settings["preparation"]:
        raise ValueError("Original cleaning/eligibility settings do not match the model.")
    candidates = []
    for path in (cfg.PROJECT_ROOT / "12_longer_history_experiment").glob("*/experiment_manifest.json"):
        record = json.loads(path.read_text())
        protocol = record["protocol"]
        if (protocol["candidate"] == selection["candidate"] and protocol["cutoff"] == selection["score_cutoff"]
                and protocol["input_sha256"].get(str(original_path.resolve())) == manifest["model_sha256"]
                and record.get("prepared_test_accessed") is False):
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError("No compatible completed longer-history experiment found. Complete 12 first.")
    candidate_manifest_path = max(candidates, key=lambda path: path.stat().st_mtime_ns)
    candidate_manifest = json.loads(candidate_manifest_path.read_text())
    for filename, expected in candidate_manifest["output_sha256"].items():
        path = candidate_manifest_path.parent / filename
        if not path.is_file() or file_hash(path) != expected:
            raise ValueError("Longer-history experiment artifacts changed. Restore their originals.")
    for filename, expected in candidate_manifest["protocol"]["input_sha256"].items():
        if not Path(filename).is_file() or file_hash(Path(filename)) != expected:
            raise ValueError("Longer-history source/provenance changed. Restore the registered candidate's inputs.")
    longer_path = candidate_manifest_path.parent / "longer_history_model.joblib"
    packages = candidate_manifest["protocol"]["package_versions"]
    for package, expected in packages.items():
        if metadata.version(package) != expected:
            raise ValueError(f"Use the recorded {package} version before registering these model environments.")
    models = {}
    registration_dir.mkdir(parents=True, exist_ok=True)
    for name, source in [("original_2025", original_path), ("longer_2024_2025", longer_path)]:
        target = registration_dir / f"{name}.joblib"
        expected = file_hash(source)
        temporary = target.with_suffix(".tmp")
        shutil.copyfile(source, temporary)
        if file_hash(temporary) != expected:
            raise ValueError("Copied model does not match its original checkpoint.")
        temporary.replace(target)
        models[name] = {"source": str(source.resolve()), "frozen_file": target.name,
            "sha256": expected, "score_cutoff": selection["score_cutoff"]}
    schedule = []
    for start in [OUTCOME_START + timedelta(days=7 * i) for i in range(4)]:
        schedule.append({"forecast_week_start": str(start - timedelta(days=7)),
            "forecast_week_end": str(start - timedelta(days=1)),
            "outcome_week_start": str(start), "outcome_week_end": str(start + timedelta(days=6))})
    registration = {"design": design, "registered_at": now.isoformat(),
        "status": "registered; no future outcomes loaded or evaluated", "models": models,
        "selected_candidate": selection["candidate"], "modelling_settings": settings,
        "frozen_excluded_item_codes": cleaning["excluded_item_codes"],
        "cleaning_sha256": file_hash(cleaning_path), "package_versions": packages,
        "longer_experiment_manifest": str(candidate_manifest_path.resolve()),
        "longer_experiment_sha256": file_hash(candidate_manifest_path),
        "forecast_schedule": schedule, "outcomes_loaded": False}
    registration["registration_sha256"] = canonical_hash(registration)
    temporary = registration_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(registration, indent=2))
    temporary.replace(registration_path)
    print("Registered the two fixed model copies; original experiments are unchanged.")
print("Registration:", registration_path)
print("Registered at:", registration["registered_at"])
print("Later-history checkpoint:", registration["models"]["longer_2024_2025"]["source"])
for name, record in registration["models"].items():
    print(name, "| SHA-256:", record["sha256"], "| frozen cutoff:", record["score_cutoff"])
print("Reserved outcome weeks:")
for row in registration["forecast_schedule"]:
    print(row["outcome_week_start"], "to", row["outcome_week_end"], "| forecast after", row["forecast_week_end"])
print("Earliest planned evaluation:", registration["design"]["earliest_evaluation"])
print("No price rows, labels, predictions or evaluation metrics were loaded.")

Registered the two fixed model copies; original experiments are unchanged.
Registration: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/13_registered_future_evaluation/2026-11-02_2026-11-29/registration.json
Registered at: 2026-10-20T01:11:36.783828+08:00
Later-history checkpoint: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/12_longer_history_experiment/3a31a8d36aad3a04/longer_history_model.joblib
original_2025 | SHA-256: bbc733f1e495ac9bc36e47b8748c523811f24c7d0364811776e0aad07b0c81df | frozen cutoff: 0.6006503045679725
longer_2024_2025 | SHA-256: e8425a04b82f7480e66e4c12ebe748bf8d259371521bf6ad1a67223f58355054 | frozen cutoff: 0.6006503045679725
Reserved outcome weeks:
2026-11-02 to 2026-11-08 | forecast after 2026-11-01
2026-11-09 to 2026-11-15 | forecast after 2026-11-08
2026-11-16 to 2026-11-22 | forecast after 2026-11-15
2026-11-23 to 2026-11-29 | forecast after 2026-11-22
Earliest planned evaluation: 2026-11-30
No price r

## What happens next

Keep this registration and both model copies unchanged. There are **no new performance results yet**. An evaluation stage must use this registered protocol, verify the model hashes, audit newly available source data, build past-only features and evaluate both versions on the same supported rows. It must record source versions, metadata exclusions and coverage rather than silently substitute stale or missing outcomes.

This notebook deliberately does not download November prices or provide a switch for early inspection. A separate implementation can be prepared before the period finishes, but no outcome evaluation should run before the planned date. The later source snapshot can still contain publisher revisions: registration alone is not proof that predictions were issued live before their outcomes.

If your FYP deadline is earlier, proceed with the dashboard and report the longer-history improvement as exploratory validation evidence. Do not claim that July–September is untouched when exposure is uncertain, and do not wait for future results just to finish the prototype.